In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.linear_model import Lasso, Ridge
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from scipy.stats import skew
from scipy.special import boxcox1p

# 1. 沿用之前的加载与清洗逻辑
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
train = train.drop(train[(train['GrLivArea']>4000) & (train['SalePrice']<300000)].index)
y = np.log1p(train['SalePrice'])
all_data = pd.concat((train, test)).drop(['Id', 'SalePrice'], axis=1)

# 2. 精简特征工程（减少噪音）
# 在高分阶段，特征越少越稳。我们只保留最核心的构造特征。
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['HasGarage'] = all_data['GarageArea'].apply(lambda x: 1 if x > 0 else 0)

# 3. 分支处理数据
# --- 分支 A: 线性模型用 (One-Hot) ---
all_data_a = all_data.copy()
num_cols = all_data_a.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols: all_data_a[col] = pd.to_numeric(all_data_a[col], errors='coerce').fillna(0)
for col in all_data_a.select_dtypes(include=['object']).columns: all_data_a[col] = all_data_a[col].fillna("None")
for feat in num_cols:
    if abs(skew(all_data_a[feat])) > 0.75:
        all_data_a[feat] = boxcox1p(all_data_a[feat], 0.15)
all_data_a = pd.get_dummies(all_data_a)
X_a = all_data_a[:len(y)]; X_test_a = all_data_a[len(y):]

# --- 分支 B: 树模型用 (Category) ---
all_data_b = all_data.copy()
cat_features = all_data_b.select_dtypes(include=['object']).columns.tolist()
for col in cat_features: all_data_b[col] = all_data_b[col].fillna('None').astype('category')
for col in all_data_b.select_dtypes(exclude=['category']).columns: all_data_b[col] = all_data_b[col].fillna(0)
X_b = all_data_b[:len(y)]; X_test_b = all_data_b[len(y):]

# 4. 训练模型（控制迭代次数防止过拟合）
print("正在训练平衡版模型...")
# 线性模型（提供稳定性）
model_lasso = Lasso(alpha=0.0005, random_state=42).fit(X_a, y)
# 树模型（提供精度）
model_cat = CatBoostRegressor(iterations=2000, learning_rate=0.05, depth=4, silent=True).fit(X_a, y)

# LGBM 使用之前 0.119 的强力参数，但缩短迭代次数
lgbm_params = {'objective':'regression','learning_rate':0.01,'num_leaves':15,'max_depth':6,
               'feature_fraction':0.2,'bagging_fraction':0.75,'bagging_freq':5,'verbose':-1}
dtrain = lgb.Dataset(X_b, label=y, categorical_feature=cat_features)
model_lgb = lgb.train(lgbm_params, dtrain, num_boost_round=2200) # 减少迭代次数防止过拟合

# 5. 权重再平衡（这是提分关键）
# 提高线性模型的权重，它能修正树模型在测试集上的偏差
pred_lasso = model_lasso.predict(X_test_a)
pred_cat = model_cat.predict(X_test_a)
pred_lgbm = model_lgb.predict(X_test_b)

# 组合：LGBM(50%) + CatBoost(30%) + Lasso(20%)
# 这种组合在小样本上极其稳健
final_pred = (0.50 * pred_lgbm) + (0.30 * pred_cat) + (0.20 * pred_lasso)

# 6. 还原与提交
submission = pd.DataFrame({'Id': test['Id'], 'SalePrice': np.expm1(final_pred)})
submission.to_csv('submission_v3.csv', index=False)
print("✅ 文件已生成。")